# Credit Default 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 Credit Default 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [2]:
CONFIG_PATH = "configs/credit_default_bttwd.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = True

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/credit_default_bttwd.yaml
运行消融实验： True


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """Read and display the current dataset summary for one run mode."""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} results =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("Reading:", dataset_summary)
        summary_df = pd.read_csv(dataset_summary)
        if "dataset_name" in summary_df.columns:
            dataset_row = summary_df[summary_df["dataset_name"].astype(str) == config_stem]
            if dataset_row.empty:
                available = ", ".join(summary_df["dataset_name"].astype(str).tolist())
                print(f"No row for {config_stem}; available datasets: {available}")
            else:
                display(dataset_row.reset_index(drop=True))
        else:
            print("dataset_summary.csv has no dataset_name column; cannot filter by dataset.")
    else:
        print("Missing dataset_summary.csv:", dataset_summary)

    if fold_summary.exists():
        print("Reading:", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("Missing fold_summary.csv:", fold_summary)

    if sample_records.exists():
        print("Sample records:", sample_records)
    else:
        print("Missing sample_records.csv:", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/credit_default_bttwd.yaml
标准输出：
【INFO】【2026-05-18 19:34:52】【数据加载】目标列 default payment next month 已检测为 0/1 标签，跳过映射逻辑
【INFO】【2026-05-18 19:34:52】【数据集信息】名称=credit_default，样本数=30000，目标列=default payment next month，正类比例=22.12%
【INFO】【2026-05-18 19:34:52】【预处理】缺失值填充策略=median
【INFO】【2026-05-18 19:34:52】已生成 credit_default 派生特征：ever_delay / max_delay / max_delay_bin
【INFO】【2026-05-18 19:34:52】ever_delay 分布：
ever_delay
0    19931
1    10069
【INFO】【2026-05-18 19:34:52】max_delay_bin 分布：
max_delay_bin
0      19931
1-2     8876
3-4     1007
5+       186
【INFO】【2026-05-18 19:34:52】max_delay_bins=[-0.1, 0, 2, 4, 9], labels=['0', '1-2', '3-4', '5+']
【INFO】【2026-05-18 19:34:52】【预处理】连续特征=14个，类别特征=5个
【INFO】【2026-05-18 19:34:52】【预处理】编码后维度=33
【INFO】【2026-05-18 19:34:52】【governance】credit_default_bttwd 第 1/5 折
【INFO】【2026-05-18 19:34:53】【桶树】已为样本生成桶ID，共 12 个组合
【INFO】【2026-05-18 19:34:53】【桶树】已为样本生成桶ID，共 12 个组合
【INFO】【2026-05-18 1

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,credit_default_bttwd,0.4165,0.705094,0.524451,0.748033,0.404767,0.670032,0.488058,0.784,1.0,...,0.472222,0.450549,0.549451,0.527778,0.527559,0.283465,0.244094,0.021673,50.0,57.0


Reading: outputs\governance\full\credit_default_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,credit_default_bttwd,1,0.407667,0.711892,0.532792,0.750667,0.392708,0.675710,0.497817,0.789167,...,0.307692,0.695652,0.304348,0.692308,0.305556,0.361111,-0.055556,-0.387960,7.0,27.0
1,credit_default_bttwd,2,0.425833,0.697662,0.515800,0.747167,0.413167,0.664280,0.479200,0.783000,...,0.750000,0.411765,0.588235,0.250000,0.619048,0.190476,0.428571,0.338235,10.0,3.0
2,credit_default_bttwd,3,0.412500,0.708119,0.528065,0.749167,0.402333,0.665908,0.481423,0.781333,...,0.500000,0.300000,0.700000,0.500000,0.642857,0.285714,0.357143,0.200000,7.0,6.0
3,credit_default_bttwd,4,0.407333,0.711325,0.533333,0.755000,0.400750,0.668235,0.486553,0.790000,...,0.636364,0.357143,0.642857,0.363636,0.641026,0.282051,0.358974,0.279221,18.0,12.0
4,credit_default_bttwd,5,0.429167,0.696471,0.512263,0.738167,0.414875,0.676025,0.495295,0.776500,...,0.250000,0.384615,0.615385,0.750000,0.529412,0.235294,0.294118,-0.134615,8.0,9.0


Sample records: outputs\governance\full\credit_default_bttwd\sample_records.csv


## 运行无 CP 消融


In [5]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/credit_default_bttwd.yaml --disable-cp-validation
标准输出：
【INFO】【2026-05-18 19:36:27】【数据加载】目标列 default payment next month 已检测为 0/1 标签，跳过映射逻辑
【INFO】【2026-05-18 19:36:27】【数据集信息】名称=credit_default，样本数=30000，目标列=default payment next month，正类比例=22.12%
【INFO】【2026-05-18 19:36:27】【预处理】缺失值填充策略=median
【INFO】【2026-05-18 19:36:27】已生成 credit_default 派生特征：ever_delay / max_delay / max_delay_bin
【INFO】【2026-05-18 19:36:27】ever_delay 分布：
ever_delay
0    19931
1    10069
【INFO】【2026-05-18 19:36:27】max_delay_bin 分布：
max_delay_bin
0      19931
1-2     8876
3-4     1007
5+       186
【INFO】【2026-05-18 19:36:27】max_delay_bins=[-0.1, 0, 2, 4, 9], labels=['0', '1-2', '3-4', '5+']
【INFO】【2026-05-18 19:36:27】【预处理】连续特征=14个，类别特征=5个
【INFO】【2026-05-18 19:36:27】【预处理】编码后维度=33
【INFO】【2026-05-18 19:36:27】【governance】credit_default_bttwd 第 1/5 折
【INFO】【2026-05-18 19:36:28】【桶树】已为样本生成桶ID，共 12 个组合
【INFO】【2026-05-18 19:36:28】【桶树】已为样本生成桶ID，共 12

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,avg_bnd_early_rescue_positive_rate,avg_non_root_bnd_rescue_rate,avg_root_bnd_reduction_rate,avg_rescue_vs_root_error_delta,avg_rescue_vs_root_regret_delta,avg_bnd_rescue_parent_candidate_success_rate,avg_bnd_rescue_from_effective_root_error_rate,bnd_early_rescue_condition_counts,bnd_early_rescue_condition_error_rates,bnd_early_rescue_condition_regrets
0,credit_default_bttwd,0.4158,0.705499,0.525133,0.749,0.404767,0.670032,0.488058,0.784,1.0,...,0.646497,0.961851,0.96361,0.017843,0.170698,0.889572,0.57734,"{""posterior_margin+bucket_margin"": 325, ""poste...","{""posterior_margin+bucket_margin"": 0.178461538...","{""posterior_margin+bucket_margin"": 0.535384615..."


Reading: outputs\governance\no_cp\credit_default_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,credit_default_bttwd,1,0.407167,0.712268,0.533250,0.750833,0.392708,0.675710,0.497817,0.789167,...,0.307692,0.695652,0.304348,0.692308,0.305556,0.361111,-0.055556,-0.387960,7.0,27.0
1,credit_default_bttwd,2,0.426833,0.696797,0.514890,0.747500,0.413167,0.664280,0.479200,0.783000,...,0.750000,0.411765,0.588235,0.250000,0.619048,0.190476,0.428571,0.338235,10.0,3.0
2,credit_default_bttwd,3,0.412333,0.708226,0.528231,0.749333,0.402333,0.665908,0.481423,0.781333,...,0.500000,0.300000,0.700000,0.500000,0.642857,0.285714,0.357143,0.200000,7.0,6.0
3,credit_default_bttwd,4,0.407000,0.711594,0.533629,0.755000,0.400750,0.668235,0.486553,0.790000,...,0.428571,0.434783,0.565217,0.571429,0.533333,0.233333,0.300000,-0.006211,13.0,12.0
4,credit_default_bttwd,5,0.425667,0.698606,0.515664,0.742333,0.414875,0.676025,0.495295,0.776500,...,0.250000,0.384615,0.615385,0.750000,0.529412,0.235294,0.294118,-0.134615,8.0,9.0


Sample records: outputs\governance\no_cp\credit_default_bttwd\sample_records.csv


## 运行无渐进更新消融


In [6]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/credit_default_bttwd.yaml --disable-progressive-update
标准输出：
【INFO】【2026-05-18 19:37:59】【数据加载】目标列 default payment next month 已检测为 0/1 标签，跳过映射逻辑
【INFO】【2026-05-18 19:37:59】【数据集信息】名称=credit_default，样本数=30000，目标列=default payment next month，正类比例=22.12%
【INFO】【2026-05-18 19:37:59】【预处理】缺失值填充策略=median
【INFO】【2026-05-18 19:37:59】已生成 credit_default 派生特征：ever_delay / max_delay / max_delay_bin
【INFO】【2026-05-18 19:37:59】ever_delay 分布：
ever_delay
0    19931
1    10069
【INFO】【2026-05-18 19:37:59】max_delay_bin 分布：
max_delay_bin
0      19931
1-2     8876
3-4     1007
5+       186
【INFO】【2026-05-18 19:37:59】max_delay_bins=[-0.1, 0, 2, 4, 9], labels=['0', '1-2', '3-4', '5+']
【INFO】【2026-05-18 19:37:59】【预处理】连续特征=14个，类别特征=5个
【INFO】【2026-05-18 19:37:59】【预处理】编码后维度=33
【INFO】【2026-05-18 19:37:59】【governance】credit_default_bttwd 第 1/5 折
【INFO】【2026-05-18 19:38:00】【桶树】已为样本生成桶ID，共 12 个组合
【INFO】【2026-05-18 19:38:00】【桶树】已为样本生成桶ID

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,avg_bnd_early_rescue_positive_rate,avg_non_root_bnd_rescue_rate,avg_root_bnd_reduction_rate,avg_rescue_vs_root_error_delta,avg_rescue_vs_root_regret_delta,avg_bnd_rescue_parent_candidate_success_rate,avg_bnd_rescue_from_effective_root_error_rate,bnd_early_rescue_condition_counts,bnd_early_rescue_condition_error_rates,bnd_early_rescue_condition_regrets
0,credit_default_bttwd,0.4165,0.705094,0.524451,0.748033,0.404767,0.670032,0.488058,0.784,1.0,...,0.646497,0.961851,0.961851,0.018603,0.161884,0.889572,0.57734,"{""posterior_margin+bucket_margin"": 325, ""poste...","{""posterior_margin+bucket_margin"": 0.178461538...","{""posterior_margin+bucket_margin"": 0.535384615..."


Reading: outputs\governance\no_progressive\credit_default_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,credit_default_bttwd,1,0.407667,0.711892,0.532792,0.750667,0.392708,0.675710,0.497817,0.789167,...,0.307692,0.695652,0.304348,0.692308,0.305556,0.361111,-0.055556,-0.387960,7.0,27.0
1,credit_default_bttwd,2,0.425833,0.697662,0.515800,0.747167,0.413167,0.664280,0.479200,0.783000,...,0.750000,0.411765,0.588235,0.250000,0.619048,0.190476,0.428571,0.338235,10.0,3.0
2,credit_default_bttwd,3,0.412500,0.708119,0.528065,0.749167,0.402333,0.665908,0.481423,0.781333,...,0.500000,0.300000,0.700000,0.500000,0.642857,0.285714,0.357143,0.200000,7.0,6.0
3,credit_default_bttwd,4,0.407333,0.711325,0.533333,0.755000,0.400750,0.668235,0.486553,0.790000,...,0.636364,0.357143,0.642857,0.363636,0.641026,0.282051,0.358974,0.279221,18.0,12.0
4,credit_default_bttwd,5,0.429167,0.696471,0.512263,0.738167,0.414875,0.676025,0.495295,0.776500,...,0.250000,0.384615,0.615385,0.750000,0.529412,0.235294,0.294118,-0.134615,8.0,9.0


Sample records: outputs\governance\no_progressive\credit_default_bttwd\sample_records.csv
